# Phase 2.1：Token、TF、IDF 与手写 BM25

## 目标

先把文字变成 token，再从词频、文档频率推导 BM25 的直觉。最后用一个教学版 BM25 和项目正式 `BM25Retriever` 对照。

**本课交付：** `data/processed/phase2_bm25_baseline.json`。

## Evidence Quest 任务卡：Phase 2.1：关键词搜索擂台

**你的身份：** 搜索擂台选手  
**案件背景：** 第一场比赛是 BM25。它不会读心，却能清楚告诉你哪些词命中了、稀有词为什么更有价值。

### 本关专业 Goal

从零理解 token、TF、DF、IDF，并建立可解释的 BM25 baseline。

### 你要交付的作品

**BM25 搜索排行榜 + 命中词解释**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：关键词侦探  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. 检索器为什么不能直接比较整句字符串？

Query 和 Document 都是字符串，但检索需要知道“哪些词出现、出现多少次、在哪些文档出现”。Tokenizer 把字符串转换成 token 列表，后续统计才有输入。

本项目的中文 baseline 用单个汉字和英文/数字词，优点是确定、无额外词典；缺点是中文专业词可能被拆开。这个取舍要用 qrels 验证。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase2.1'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase2.1
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 从项目 BM25 模块导入确定性 tokenizer。
from phase2_semantic_search.bm25 import tokenize

# 准备一条同时包含中文、英文和数字的 Query。
query_text = "BM25 对产品型号更稳 2024"

# 调用 tokenizer，把 Query 转为可统计的 token 列表。
query_tokens = tokenize(query_text)

# 打印原文和 token，观察中文被如何切分。
print("原文:", query_text)
print("tokens:", query_tokens)

# 确认 tokenizer 至少产生了一个 token。
assert query_tokens

原文: BM25 对产品型号更稳 2024
tokens: ['bm25', '对', '产', '品', '型', '号', '更', '稳', '2024']


## 2. 从最简单的 TF 和 DF 开始

- **TF（Term Frequency）**：一个词在当前文档出现几次。
- **DF（Document Frequency）**：一个词出现在多少篇不同文档。
- **IDF**：DF 越低，词越稀有，越能区分文档。

先用小语料手算，之后再看完整 BM25 公式。

In [4]:
# 定义三篇极小文档，每篇文档用 token 列表表示。
toy_documents = [["bm25", "search", "bm25"], ["dense", "search"], ["search", "api"]]

# 选择要观察的词。
target_term = "bm25"

# 统计目标词在第一篇文档中的出现次数。
term_frequency = toy_documents[0].count(target_term)

# 统计目标词出现在多少篇不同文档中。
document_frequency = sum(target_term in document for document in toy_documents)

# 打印 TF 和 DF，观察“当前文档频率”和“全局文档频率”的区别。
print("TF:", term_frequency)
print("DF:", document_frequency)

# 目标词应该在第一篇文档出现两次。
assert term_frequency == 2

# 目标词只应该出现在一篇文档中。
assert document_frequency == 1

TF: 2
DF: 1


In [5]:
# 导入数学库，用对数计算 IDF。
import math

# 记录 toy corpus 的文档总数。
document_count = len(toy_documents)

# 使用 BM25 常见的平滑公式计算 IDF。
idf_value = math.log(1 + (document_count - document_frequency + 0.5) / (document_frequency + 0.5))

# 输出 IDF，观察稀有词会获得较高的区分权重。
print("IDF:", round(idf_value, 4))

# 用一个所有文档都有的词计算对照 IDF。
common_frequency = sum("search" in document for document in toy_documents)

# 计算 common term 的 IDF。
common_idf = math.log(1 + (document_count - common_frequency + 0.5) / (common_frequency + 0.5))

# 对比稀有词和常见词的权重。
print("common term IDF:", round(common_idf, 4))
print("稀有词权重更高:", idf_value > common_idf)

IDF: 0.9808
common term IDF: 0.1335
稀有词权重更高: True


## 3. 手写一个可读的 BM25 版本

完整公式包含 TF 饱和和长度归一化。教学版本只支持一个 Query 和一个文档列表，但保留三个核心思想：命中奖励、IDF、长度修正。每条语句都写注释，先关注变量如何流动。

In [6]:
# 定义一个教学版 BM25 函数，输入是 Query token 和文档 token 列表。
def simple_bm25(query_terms, documents, k1=1.5, b=0.75):
    # 导入 Counter，方便统计每篇文档的词频。
    from collections import Counter

    # 记录整个语料的文档数量。
    total_documents = len(documents)

    # 计算所有文档的平均 token 长度。
    average_length = sum(len(document) for document in documents) / max(total_documents, 1)

    # 统计每个词出现在多少篇不同文档中。
    document_frequencies = Counter(term for document in documents for term in set(document))

    # 创建空列表，用于保存每篇文档的最终得分。
    scores = []

    # 逐篇文档计算 Query 的匹配分数。
    for document in documents:
        # 统计当前文档中每个词的出现次数。
        frequencies = Counter(document)

        # 记录当前文档的 token 长度。
        document_length = len(document)

        # 从零开始累加当前文档得分。
        score = 0.0

        # 逐个 Query token 计算贡献。
        for term in set(query_terms):
            # 读取该词在当前文档中的 TF。
            term_frequency = frequencies.get(term, 0)

            # 没有命中时，该词对当前文档没有贡献。
            if term_frequency == 0:
                continue

            # 读取该词的 DF。
            term_document_frequency = document_frequencies[term]

            # 计算该词的 IDF。
            term_idf = math.log(1 + (total_documents - term_document_frequency + 0.5) / (term_document_frequency + 0.5))

            # 计算文档长度归一化项。
            length_factor = 1 - b + b * document_length / max(average_length, 1e-12)

            # 计算 TF 饱和后的贡献并累加。
            score += term_idf * term_frequency * (k1 + 1) / (term_frequency + k1 * length_factor)

        # 保存当前文档的总分。
        scores.append(score)

    # 返回每篇文档的分数，顺序与输入文档一致。
    return scores

# 对 toy corpus 运行教学版 BM25。
toy_scores = simple_bm25(["bm25"], toy_documents)

# 打印得分，观察命中文档和未命中文档的区别。
print(toy_scores)

# 第一篇文档命中了 bm25，应该获得正分。
assert toy_scores[0] > 0

# 第二篇文档没有命中 bm25，应该得分为零。
assert toy_scores[1] == 0

[1.2833279945947822, 0.0, 0.0]


### 为什么生产实现比教学函数长？

教学函数只返回分数，生产 `BM25Retriever` 还要处理 tokenizer、top-k、过滤器、稳定排序、文档元数据和输入校验。理解公式后再使用模块，你能知道这些工程代码分别保护什么行为。

In [7]:
# 读取 Phase 1 的真实 Chunk 数据。
chunks_path = ROOT / "data" / "processed" / "chunks.json"
chunks = json.loads(chunks_path.read_text(encoding="utf-8"))

# 导入生产 BM25 检索器。
from phase2_semantic_search.bm25 import BM25Retriever

# 用真实 Chunk 创建索引对象。
retriever = BM25Retriever(chunks)

# 运行一条真实 Query。
real_results = retriever.search("Chunk overlap", top_k=5)

# 输出排名、分数、来源和正文片段，观察生产结果合同。
for rank, result in enumerate(real_results, start=1):
    # 每一行代表一个可追溯检索结果。
    print(rank, result.doc_id, round(result.score, 4), result.metadata.get("source"), result.text[:80])

# 至少应该找到一条包含 Query 词的结果。
assert real_results

1 82ddc7d612e87b02 1.746 D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input\quickstart.md 据，让后续检索结果可以回溯到原文。

## Chunk 策略

先按段落和换行切分，再按中文标点递归降级。overlap 用于保留跨边界的上下文，但会增加索引体
2 3692b05e025373a8 0.7116 D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input\quickstart.md # Phase 1 Quickstart

文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。

## Chunk 策


In [8]:
# 创建 baseline 记录，保存 Query 和结果 ID，而不是只保存屏幕输出。
baseline_record = {
    "retriever": "BM25",
    "query": "Chunk overlap",
    "top_k": 5,
    "results": [{"rank": rank, "chunk_id": result.doc_id, "score": result.score} for rank, result in enumerate(real_results, start=1)],
}

# 指定 baseline 的输出路径。
baseline_path = ROOT / "data" / "processed" / "phase2_bm25_baseline.json"

# 保存 baseline，供后续 RRF 和评估 Notebook 读取。
baseline_path.write_text(json.dumps(baseline_record, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印交付路径。
print("已生成:", baseline_path)

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\phase2_bm25_baseline.json


## 本课验收

- [ ] 能区分 TF、DF、IDF。
- [ ] 能解释 BM25 为什么不会无限奖励重复词。
- [ ] 能读懂手写函数中每个变量的来源和用途。
- [ ] 能把教学实现和生产实现的差异说清楚。
- [ ] 已生成 `phase2_bm25_baseline.json`。

## Boss Challenge：挑一个稀有词和一个常见词，比较它们的 IDF，并解释哪个更能区分案件。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [9]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [10]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase2_bm25_baseline.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\phase2_bm25_baseline.json']
